<a href="https://colab.research.google.com/github/treatbam/Actions-Alpine-Linux/blob/master/Tinkerboard_OLED_Monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
import time
import os
import socket
import psutil
from PIL import Image, ImageDraw, ImageFont
import smbus2

# Luma OLED imports
from luma.core.interface.serial import spi
from luma.core.render import canvas
from luma.oled.device import ssd1351

# Optional Docker support
try:
    import docker
    docker_client = docker.from_env()
except Exception:
    docker_client = None

# ==========================================
# CONFIGURATION
# ==========================================
DISPLAY_WIDTH = 128
DISPLAY_HEIGHT = 96
SPI_PORT = 0
SPI_DEVICE = 0
GPIO_DC = 25
GPIO_RST = 24
BGR_MODE = True

PAGE_DUR_SEC = 5.0
REFRESH_RATE = 0.5

# ==========================================
# UPS I2C ABSTRACTION
# ==========================================
class WaveshareUPSE:
    """
    Handles reading the exact I2C registers from the Waveshare UPS HAT (E).
    """
    def __init__(self, bus_num=1):
        self.address = 0x2D
        try:
            self.bus = smbus2.SMBus(bus_num)
            self.has_i2c = True
        except Exception as e:
            print(f"Warning: I2C bus not found: {e}. Running in Mock Mode.")
            self.has_i2c = False

    def _read_16bit(self, reg_low, reg_high, signed=False):
        """Reads two 8-bit registers and combines them into a 16-bit integer."""
        low = self.bus.read_byte_data(self.address, reg_low)
        high = self.bus.read_byte_data(self.address, reg_high)
        val = (high << 8) | low

        if signed and val > 32767:
            val -= 65536
        return val

    def read_data(self):
        if not self.has_i2c:
            return self._get_mock_data()

        try:
            # 1. Charging Register (0x02)
            status_reg = self.bus.read_byte_data(self.address, 0x02)
            is_charging = bool(status_reg & 0x80)     # BIT7
            is_vbus_powered = bool(status_reg & 0x20) # BIT5

            if is_vbus_powered and is_charging:
                status_str = "CHARGING"
            elif is_vbus_powered and not is_charging:
                status_str = "AC / FULL"
            else:
                status_str = "ON BATTERY"

            # 2. Battery Voltage (0x20, 0x21) & Current (0x22, 0x23)
            voltage_mv = self._read_16bit(0x20, 0x21)
            current_ma = self._read_16bit(0x22, 0x23, signed=True)

            # Calculate mW. (mV * mA) / 1000 = mW
            power_mw = (voltage_mv * current_ma) / 1000.0

            # 3. SOC Percent (0x24, 0x25)
            soc_pct = self._read_16bit(0x24, 0x25)

            # 4. Remaining Time
            if current_ma > 0: # Charging
                time_val = self._read_16bit(0x2A, 0x2B)
                time_str = f"{time_val}m to full" if time_val != 65535 else "--"
            else: # Discharging or Full
                time_val = self._read_16bit(0x28, 0x29)
                time_str = f"{time_val}m left" if time_val != 65535 else "--"

            # 5. Cell Voltages (0x30 to 0x37)
            cells = [
                self._read_16bit(0x30, 0x31),
                self._read_16bit(0x32, 0x33),
                self._read_16bit(0x34, 0x35),
                self._read_16bit(0x36, 0x37)
            ]

            return {
                'status': status_str,
                'soc': min(100, max(0, soc_pct)),
                'voltage_mv': voltage_mv,
                'current_ma': current_ma,
                'power_mw': power_mw,
                'time_str': time_str,
                'cells_mv': cells
            }

        except Exception as e:
            # Prevent crashes if I2C fails mid-read
            return self._get_mock_data()

    def _get_mock_data(self):
        """Simulated data so you can test the UI before wiring up the UPS"""
        import math
        t = time.time()
        soc = 50 + 50 * math.sin(t / 20.0)
        current_ma = 2000 * math.cos(t / 20.0)
        voltage_mv = 14800 + (soc * 20)

        return {
            'status': "CHARGING" if current_ma > 0 else "ON BATTERY",
            'soc': max(0, min(100, soc)),
            'voltage_mv': voltage_mv,
            'current_ma': current_ma,
            'power_mw': (voltage_mv * current_ma) / 1000.0,
            'time_str': "1h 10m left" if current_ma <= 0 else "2h 15m to full",
            'cells_mv': [3750, 3760, 3740, 3755]
        }

# ==========================================
# UI & DRAWING HELPERS
# ==========================================
def get_font(size):
    """Loads a TTF font or falls back to default."""
    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf"
    ]
    for path in font_paths:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

def draw_ring_dashboard(draw, width, height, title, value_str, percentage, ring_color, subtext=""):
    """Draws a beautiful circular dashboard element."""
    draw.rectangle((0, 0, width, height), fill=(0, 0, 0))
    margin = 12
    bbox = (margin, margin + 12, width - margin, height - margin + 12)

    # Background ring
    dim_color = (ring_color[0]//4, ring_color[1]//4, ring_color[2]//4)
    draw.arc(bbox, 135, 405, fill=dim_color, width=8)

    # Foreground ring
    if percentage > 0:
        end_angle = 135 + (270 * (max(0, min(100, percentage)) / 100.0))
        draw.arc(bbox, 135, end_angle, fill=ring_color, width=8)

    font_title = get_font(11)
    font_value = get_font(24)
    font_sub = get_font(10)

    draw.text((width//2, 8), title, font=font_title, fill=(200, 200, 200), anchor="mt")
    draw.text((width//2, height//2 + 8), value_str, font=font_value, fill=(255, 255, 255), anchor="mm")

    if subtext:
        draw.text((width//2, height - 6), subtext, font=font_sub, fill=(100, 200, 255), anchor="mb")

# ==========================================
# SYSTEM PAGES
# ==========================================
def page_cpu(draw, w, h, ups_data):
    pct = psutil.cpu_percent(interval=None)
    draw_ring_dashboard(draw, w, h, "CPU USAGE", f"{pct:.0f}%", pct, (0, 255, 255))

def page_ram(draw, w, h, ups_data):
    pct = psutil.virtual_memory().percent
    draw_ring_dashboard(draw, w, h, "RAM USAGE", f"{pct:.0f}%", pct, (255, 0, 255))

def page_net_docker(draw, w, h, ups_data):
    draw.rectangle((0, 0, w, h), fill=(0, 0, 0))
    font_title = get_font(11)
    font_val = get_font(16)

    ip = "Offline"
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
    except Exception:
        pass

    containers = "N/A"
    if docker_client:
        try:
            containers = str(len(docker_client.containers.list()))
        except Exception:
            containers = "Err"

    draw.text((w//2, 12), "IP ADDRESS", font=font_title, fill=(0, 255, 255), anchor="mm")
    draw.text((w//2, 30), ip, font=font_val, fill=(255, 255, 255), anchor="mm")

    draw.text((w//2, 60), "DOCKERS ACTIVE", font=font_title, fill=(255, 0, 255), anchor="mm")
    draw.text((w//2, 78), containers, font=font_val, fill=(255, 255, 255), anchor="mm")

# ==========================================
# UPS PAGES
# ==========================================
def page_ups_main(draw, w, h, ups_data):
    soc = ups_data.get('soc', 0)
    status = ups_data.get('status', 'UNKNOWN')
    time_str = ups_data.get('time_str', '')

    if "CHARGING" in status or "AC" in status:
        color = (0, 255, 0)
    else:
        color = (255, 128, 0) if soc > 20 else (255, 0, 0)

    draw_ring_dashboard(draw, w, h, status, f"{soc:.0f}%", soc, color, subtext=time_str)

def page_ups_power(draw, w, h, ups_data):
    draw.rectangle((0, 0, w, h), fill=(0, 0, 0))
    font_title = get_font(11)
    font_val = get_font(20)
    font_sub = get_font(12)

    power_mw = ups_data.get('power_mw', 0.0)
    voltage_mv = ups_data.get('voltage_mv', 0.0)
    current_ma = ups_data.get('current_ma', 0.0)

    draw.text((w//2, 8), "POWER DRAW", font=font_title, fill=(150, 150, 150), anchor="mt")

    # Power in mW (using + or - sign explicitly)
    color = (0, 255, 0) if current_ma >= 0 else (255, 128, 0)
    draw.text((w//2, h//2), f"{power_mw:+.0f} mW", font=font_val, fill=color, anchor="mm")

    # Bottom Subtext
    sub_text = f"{voltage_mv/1000.0:.2f}V  |  {current_ma:+.0f}mA"
    draw.text((w//2, h - 16), sub_text, font=font_sub, fill=(100, 200, 255), anchor="mm")

def page_ups_cells(draw, w, h, ups_data):
    """Displays the 4 individual cell voltages in a 2x2 grid."""
    draw.rectangle((0, 0, w, h), fill=(0, 0, 0))
    font_title = get_font(11)
    font_cell = get_font(13)

    draw.text((w//2, 8), "CELL VOLTAGES", font=font_title, fill=(200, 200, 200), anchor="mt")

    cells = ups_data.get('cells_mv', [0, 0, 0, 0])

    # Coordinates for the 2x2 grid
    positions = [
        (w//4, 45),       (3*w//4, 45),
        (w//4, 45 + 25),  (3*w//4, 45 + 25)
    ]

    for i, (mv, pos) in enumerate(zip(cells, positions)):
        volts = mv / 1000.0
        # Color code the cell health
        if volts >= 3.6:
            color = (0, 255, 0) # Healthy Green
        elif volts >= 3.3:
            color = (255, 200, 0) # Warning Yellow
        else:
            color = (255, 0, 0) # Critical Red

        text = f"{volts:.2f}V" if volts > 0 else "N/A"
        draw.text(pos, text, font=font_cell, fill=color, anchor="mm")

# ==========================================
# MAIN ROTATION LOOP
# ==========================================
def main():
    print("Starting Tinkerboard OLED Monitor...")

    try:
        serial_intf = spi(device=SPI_DEVICE, port=SPI_PORT, gpio_DC=GPIO_DC, gpio_RST=GPIO_RST)
        device = ssd1351(serial_intf, width=DISPLAY_WIDTH, height=DISPLAY_HEIGHT, bgr=BGR_MODE)
        print("SSD1351 OLED Initialized successfully.")
    except Exception as e:
        print(f"Failed to initialize OLED: {e}")
        return

    ups = WaveshareUPSE(bus_num=1)

    pages = [
        page_cpu,
        page_ram,
        page_disk,
        page_net_docker,
        page_ups_main,
        page_ups_power,
        page_ups_cells
    ]

    current_page_idx = 0

    try:
        while True:
            start_time = time.time()
            while time.time() - start_time < PAGE_DUR_SEC:
                ups_data = ups.read_data()

                with canvas(device) as draw:
                    pages[current_page_idx](draw, device.width, device.height, ups_data)

                time.sleep(REFRESH_RATE)

            current_page_idx = (current_page_idx + 1) % len(pages)

    except KeyboardInterrupt:
        device.clear()

if __name__ == "__main__":
    main()